# Explore a pretrained ResNet model

This notebook loads a pretrained ResNet-18 model, reads a sample image, and prints the top-5 predictions.

In [ ]:
import os
from pathlib import Path

import torch
from torchvision import models

from advkit.models.loader import load_model, predict
from advkit.utils.image import load_image, tensor_to_image

# The model weights are downloaded once and cached locally by torchvision.
model = load_model("resnet18")

image_path = Path("../examples/sample_images/sample.jpg")
if not image_path.exists():
    image_path = Path("examples/sample_images/sample.jpg")

# The image loader returns a normalized tensor that matches the model input format.
image_tensor = load_image(image_path)
image_tensor = image_tensor.unsqueeze(0)

# Run inference and print the top-5 predictions with confidence scores.
pred_idx, confidence = predict(model, image_tensor.squeeze(0))
print(f"Predicted class index: {pred_idx} with confidence: {confidence:.4f}")

# The model outputs logits; softmax converts them to probabilities for human-readable scores.
with torch.no_grad():
    logits = model(image_tensor)
    probabilities = torch.softmax(logits, dim=1)[0]
    top5_probs, top5_indices = torch.topk(probabilities, 5)

for rank, (prob, idx) in enumerate(zip(top5_probs, top5_indices), start=1):
    print(f"{rank}. class {idx.item()}: {prob.item():.4f}")

# Display the image after it has been converted back from a normalized tensor.
image = tensor_to_image(image_tensor.squeeze(0))
image